In [1]:
# Re-run due to state reset

import random
import pandas as pd
from datetime import datetime, timedelta
import uuid



In [2]:
# Config
HOME_LOC = (28.6139, 77.2090)  # Delhi
CITY_POOL = {
    "Paris": (48.8566, 2.3522),
    "New York": (40.7128, -74.0060),
    "Tokyo": (35.6895, 139.6917),
    "Bangalore": (12.9716, 77.5946),
    "Goa": (15.2993, 74.1240),
}
SCENE_POOL = ["monument", "food", "beach", "street", "family", "selfie"]
FACES_POOL = [{"gender": g} for g in ["male", "female"]]

# Output storage
taking_pic = []
staying_act = []
travel_coll = []



In [3]:
# Helper
def random_time_span(start_date, min_hours=1, max_hours=3):
    start_time = start_date + timedelta(hours=random.randint(8, 18))
    duration = timedelta(hours=random.randint(min_hours, max_hours))
    return start_time, start_time + duration

def random_face_data(mode):
    count = random.randint(1, 2) if mode == "solo" else random.randint(2, 5)
    return [dict(id=str(uuid.uuid4())[:6], gender=random.choice(["male", "female"]), bbox=str((10,10,50,50))) for _ in range(count)]



In [4]:
# Generate data
start_date = datetime(2023, 1, 1)
days = 365
pic_id_counter = 0
stay_id_counter = 0
coll_id_counter = 0

for week in range(52):
    week_start = start_date + timedelta(days=week*7)

    travel_type = random.choice(["none", "visit", "short", "long"])
    travel_mode = random.choice(["solo", "family"])

    if travel_type == "none":
        continue

    location = random.choice(list(CITY_POOL.keys()))
    loc_latlon = CITY_POOL[location]

    # Travel time span
    travel_days = 1 if travel_type == "visit" else (2 if travel_type == "short" else random.randint(3, 6))
    travel_start = week_start + timedelta(days=random.randint(0, 3))
    travel_end = travel_start + timedelta(days=travel_days)

    # Staying activity (1 per trip)
    stay_start = travel_start + timedelta(hours=2)
    stay_end = travel_end - timedelta(hours=2)
    stay_id = f"stay_{stay_id_counter}"
    staying_act.append({
        "id": stay_id,
        "start_time": stay_start.isoformat(),
        "end_time": stay_end.isoformat(),
        "location": location,
        "lat": loc_latlon[0],
        "lon": loc_latlon[1],
        "reason": "hotel stay"
    })
    stay_id_counter += 1

    # Taking pic activity (multiple)
    pic_ids = []
    for _ in range(random.randint(1, 3 if travel_mode == "solo" else 5)):
        span_start, span_end = random_time_span(travel_start)
        image_ids = [f"img_{uuid.uuid4().hex[:5]}" for _ in range(random.randint(2, 5))]
        scene_tags = random.sample(SCENE_POOL, random.randint(1, 3))
        faces = random_face_data(travel_mode)

        pic_id = f"pic_{pic_id_counter}"
        taking_pic.append({
            "id": pic_id,
            "start_time": span_start.isoformat(),
            "end_time": span_end.isoformat(),
            "images": ",".join(image_ids),
            "location": location,
            "lat": loc_latlon[0],
            "lon": loc_latlon[1],
            "scenes": ",".join(scene_tags),
            "faces": str(faces)
        })
        pic_ids.append(pic_id)
        pic_id_counter += 1

    # Travel collection entry
    coll_id = f"travel_{coll_id_counter}"
    travel_coll.append({
        "id": coll_id,
        "start_time": travel_start.isoformat(),
        "end_time": travel_end.isoformat(),
        "type": travel_type,
        "mode": travel_mode,
        "location": location,
        "pic_ids": ",".join(pic_ids),
        "stay_ids": stay_id
    })
    coll_id_counter += 1



In [6]:
# Create DataFrames
df_pic = pd.DataFrame(taking_pic)
df_stay = pd.DataFrame(staying_act)
df_travel = pd.DataFrame(travel_coll)




In [7]:
df_pic.head()

,id,start_time,end_time,images,location,lat,lon,scenes,faces
0,pic_0,2023-01-02T13:00:00,2023-01-02T14:00:00,"img_3baaf,img_3e7af,img_8b975,img_bde2d,img_cf5ed",New York,40.7128,-74.0060,"monument,family,food","[{'id': '4d9988', 'gender': 'female', 'bbox': ..."
1,pic_1,2023-01-02T16:00:00,2023-01-02T18:00:00,"img_5e6b3,img_f944e,img_71791,img_f24a5,img_8a952",New York,40.7128,-74.0060,"monument,selfie","[{'id': '923aa5', 'gender': 'female', 'bbox': ..."
2,pic_2,2023-01-02T11:00:00,2023-01-02T13:00:00,"img_81793,img_61c0b,img_5d158",New York,40.7128,-74.0060,beach,"[{'id': '340749', 'gender': 'female', 'bbox': ..."
3,pic_3,2023-01-16T14:00:00,2023-01-16T16:00:00,"img_7ffb8,img_28d52,img_c60b3",Paris,48.8566,2.3522,"beach,family","[{'id': '431f05', 'gender': 'male', 'bbox': '(..."
4,pic_4,2023-01-16T16:00:00,2023-01-16T18:00:00,"img_a952b,img_c0cae,img_336df",Paris,48.8566,2.3522,"monument,food","[{'id': 'badf1b', 'gender': 'female', 'bbox': ..."


In [8]:
df_stay.head()

,id,start_time,end_time,location,lat,lon,reason
0,stay_0,2023-01-02T02:00:00,2023-01-07T22:00:00,New York,40.7128,-74.0060,hotel stay
1,stay_1,2023-01-16T02:00:00,2023-01-17T22:00:00,Paris,48.8566,2.3522,hotel stay
2,stay_2,2023-01-23T02:00:00,2023-01-23T22:00:00,Goa,15.2993,74.1240,hotel stay
3,stay_3,2023-01-30T02:00:00,2023-01-30T22:00:00,Bangalore,12.9716,77.5946,hotel stay
4,stay_4,2023-02-06T02:00:00,2023-02-07T22:00:00,New York,40.7128,-74.0060,hotel stay


In [9]:
df_travel.head()

,id,start_time,end_time,type,mode,location,pic_ids,stay_ids
0,travel_0,2023-01-02T00:00:00,2023-01-08T00:00:00,long,family,New York,"pic_0,pic_1,pic_2",stay_0
1,travel_1,2023-01-16T00:00:00,2023-01-18T00:00:00,short,solo,Paris,"pic_3,pic_4",stay_1
2,travel_2,2023-01-23T00:00:00,2023-01-24T00:00:00,visit,solo,Goa,"pic_5,pic_6,pic_7",stay_2
3,travel_3,2023-01-30T00:00:00,2023-01-31T00:00:00,visit,family,Bangalore,pic_8,stay_3
4,travel_4,2023-02-06T00:00:00,2023-02-08T00:00:00,short,family,New York,pic_9,stay_4


In [7]:
# --- Save to CSV ---
def save_csv(filename, data, header):
    with open(f"./{filename}", "w", newline='') as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        for row in data:
            writer.writerow(row)


save_csv("taking_pic.csv", taking_pic_data, ["date", "activity_id", "scene", "faces", "images", "location"])
save_csv("staying_activity.csv", staying_data, ["start_date", "end_date", "location", "duration_days"])
save_csv("travel_collection.csv", travel_collections, ["start_date", "end_date", "location", "is_family", "activities"])


A moment is a temporally and semantically localized segment of activity, like a distinct phase of a travel or event.

A moment is a temporal subgraph.

Each moment is a semantic + temporal unit (e.g., Eiffel → Boat ride → Dinner).

Change in subgraph similarity → new moment begins.

This fits naturally into your existing TKG + subgraph evolution framework.

In [12]:
from geopy.distance import geodesic
import pandas as pd
from datetime import datetime, timedelta

def compute_distance(loc1, loc2):
    return geodesic(loc1, loc2).km

def detect_moments(df_travel, df_pic, time_gap_hours=2, loc_threshold_km=2):
    moment_results = []

    for idx, row in df_travel.iterrows():
        tid = row["id"]
        start = pd.to_datetime(row["start_time"])
        end = pd.to_datetime(row["end_time"])

        # Filter pic activities within this travel
        df_segment = df_pic[
            (pd.to_datetime(df_pic["start_time"]) >= start) &
            (pd.to_datetime(df_pic["end_time"]) <= end)
        ].sort_values("start_time").reset_index(drop=True)

        moment_id = 0
        current_moment = []

        for i, act in df_segment.iterrows():
            act_time = pd.to_datetime(act["start_time"])
            act_loc = (act["lat"], act["lon"])

            if not current_moment:
                current_moment.append((act_time, act_loc, act))
                continue

            prev_time, prev_loc, _ = current_moment[-1]
            time_gap = (act_time - prev_time).total_seconds() / 3600
            dist = compute_distance(prev_loc, act_loc)

            if time_gap > time_gap_hours or dist > loc_threshold_km:
                # save current moment
                moment_results.append({
                    "travel_id": tid,
                    "moment_id": moment_id,
                    "start_time": current_moment[0][0],
                    "end_time": current_moment[-1][0],
                    "photo_count": len(current_moment),
                    "locs": [loc for _, loc, _ in current_moment],
                    "titles": [act["title"] for _, _, act in current_moment],
                })
                moment_id += 1
                current_moment = []

            current_moment.append((act_time, act_loc, act))

        # Save last moment if exists
        if current_moment:
            moment_results.append({
                "travel_id": tid,
                "moment_id": moment_id,
                "start_time": current_moment[0][0],
                "end_time": current_moment[-1][0],
                "photo_count": len(current_moment),
                "locs": [loc for _, loc, _ in current_moment],
                "titles": [act["title"] for _, _, act in current_moment],
            })

    return pd.DataFrame(moment_results)


In [13]:
moments_df = detect_moments(df_travel, df_pic)
moments_df.head()


KeyError: 'title'